In [9]:
import pandas as pd
from pathlib import Path

# 1. Definir os arquivos
arquivo_entrada = Path("05_PENSE2024_SEM_NAN_SEM_DUPLICADOS.parquet")
arquivo_saida = Path("06_PENSE2024_SEM_RESPOSTA_ABANDONO_PULO.parquet")

# 2. Carregar a base
df = pd.read_parquet(arquivo_entrada)

# 3. Definir os códigos que serão eliminados
codigos_invalidos = [-9, -2, -1]

# 4. Identificar todas as células com esses valores
mascara_invalidos = df.isin(codigos_invalidos)

# 5. Identificar as linhas que possuem pelo menos um código inválido
mascara_excluir = mascara_invalidos.any(axis=1)

# 6. Excluir integralmente essas linhas
df_limpo = df.loc[~mascara_excluir].copy()

# 7. Calcular os resultados
total_original = len(df)
total_excluido = int(mascara_excluir.sum())
total_restante = len(df_limpo)

percentual_removido = (total_excluido / total_original) * 100

# 8. Exibir o resultado
print("========== RESULTADO DA LIMPEZA ==========")

print(f"Linhas originais: {total_original:,}")
print(f"Linhas excluídas: {total_excluido:,}")
print(f"Linhas restantes: {total_restante:,}")
print(f"Percentual removido: {percentual_removido:.2f}%")

# 9. Verificar se ainda existem códigos inválidos
restantes_invalidos = int(df_limpo.isin(codigos_invalidos).sum().sum())

print(f"Códigos inválidos restantes: {restantes_invalidos}")

# 10. Salvar a nova base
df_limpo.to_parquet(arquivo_saida, index=False)

print(f"\nArquivo salvo: {arquivo_saida}")

========== RESULTADO DA LIMPEZA ==========
Linhas originais: 148,707
Linhas excluídas: 0
Linhas restantes: 148,707
Percentual removido: 0.00%
Códigos inválidos restantes: 0

Arquivo salvo: 06_PENSE2024_SEM_RESPOSTA_ABANDONO_PULO.parquet


In [10]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Carregar o dicionário de dados
dicionario = pd.read_excel("00_Dicionário_Microdados_PENSE2024.xlsx")

dico_map = {}
for _, row in dicionario.iterrows():
    desc = str(row["Descrição da Variável"]).strip()
    cat = str(row["Categoria"]) if pd.notna(row["Categoria"]) else ""
    cat_dict = {}
    if cat:
        for linha in cat.split("\n"):
            if " - " in linha:
                partes = linha.split(" - ", 1)
                cat_dict[partes[0].strip()] = partes[1].strip()
    dico_map[desc] = cat_dict

# 2. Gerar a tabela em Markdown
total = len(df_limpo)
markdown_lines = [
    "| Pergunta / Variável | Distribuição das Respostas |",
    "| :--- | :--- |"
]

for col in df_limpo.columns:
    counts = df_limpo[col].astype(str).str.strip().value_counts(dropna=False)
    cat_dict = dico_map.get(col, {})
    
    respostas_formatadas = []
    for cod, qtd in counts.items():
        rotulo = cat_dict.get(cod, "")
        pct = (qtd / total) * 100
        if rotulo:
            respostas_formatadas.append(f"[{cod}] {rotulo}: {pct:.2f}%")
        else:
            respostas_formatadas.append(f"[{cod}]: {pct:.2f}%")
            
    # Subsitui quebras de linha normais por <br> do HTML para a tabela Markdown respeitar a formatação
    distribuicao_str = "<br>".join(respostas_formatadas)
    
    # Limpa quebras de linha no nome da pergunta para não quebrar a tabela Markdown
    col_clean = str(col).replace("\n", " ").replace("|", "-")
    
    markdown_lines.append(f"| {col_clean} | {distribuicao_str} |")

# 3. Exibir a tabela renderizada
print("========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========\n")
display(Markdown("\n".join(markdown_lines)))

========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========



| Pergunta / Variável | Distribuição das Respostas |
| :--- | :--- |
| NOS ÚLTIMOS 12 MESES, alguma vez você sentiu vontade de se machucar de propósito? | [2] Não: 66.75%<br>[1] Sim: 32.66%<br>[-9] Sem resposta: 0.35%<br>[-2] Abandono: 0.24% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que a vida não vale a pena ser vivida? | [1] Nunca: 48.83%<br>[2] Raramente: 17.37%<br>[3] Às vezes: 15.61%<br>[4] Na maioria das vezes: 8.80%<br>[5] Sempre: 8.74%<br>[-9] Sem resposta: 0.41%<br>[-2] Abandono: 0.23% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu triste? | [3] Às vezes: 36.24%<br>[2] Raramente: 23.71%<br>[4] Na maioria das vezes: 19.28%<br>[1] Nunca: 11.07%<br>[5] Sempre: 9.18%<br>[-9] Sem resposta: 0.31%<br>[-2] Abandono: 0.22% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que ninguém se preocupa com você? | [1] Nunca: 27.76%<br>[3] Às vezes: 24.36%<br>[2] Raramente: 22.24%<br>[4] Na maioria das vezes: 14.20%<br>[5] Sempre: 10.87%<br>[-9] Sem resposta: 0.36%<br>[-2] Abandono: 0.22% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu muito preocupado com as coisas comuns do seu dia a dia como atividades da escola, competições esportivas, tarefas de casa, etc.? | [4] Na maioria das vezes: 27.73%<br>[3] Às vezes: 25.46%<br>[5] Sempre: 25.11%<br>[2] Raramente: 11.73%<br>[1] Nunca: 9.39%<br>[-9] Sem resposta: 0.36%<br>[-2] Abandono: 0.21% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu irritado(a), nervoso(a) ou mal-humorado(a) por qualquer coisa? | [3] Às vezes: 31.85%<br>[4] Na maioria das vezes: 24.84%<br>[5] Sempre: 17.76%<br>[2] Raramente: 15.73%<br>[1] Nunca: 9.25%<br>[-9] Sem resposta: 0.35%<br>[-2] Abandono: 0.23% |
| Quantos(as) amigos(as) próximos você tem? | [4] 3 ou mais amigos: 75.58%<br>[3] 2 amigos: 13.72%<br>[2] 1 amigo: 6.41%<br>[1] Nenhum amigo: 3.86%<br>[-9] Sem resposta: 0.23%<br>[-2] Abandono: 0.20% |
| NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável entendeu suas preocupações e problemas? | [5] Sempre: 22.93%<br>[4] Na maioria das vezes: 21.73%<br>[3] Às vezes: 20.28%<br>[2] Raramente: 17.28%<br>[1] Nunca: 17.23%<br>[-9] Sem resposta: 0.41%<br>[-2] Abandono: 0.15% |
| NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre? | [5] Sempre: 33.90%<br>[4] Na maioria das vezes: 33.54%<br>[3] Às vezes: 15.27%<br>[2] Raramente: 8.85%<br>[1] Nunca: 7.96%<br>[-9] Sem resposta: 0.34%<br>[-2] Abandono: 0.13% |
| NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola o esculachou, zoou, mangou, intimidou ou caçoou tanto que você ficou magoado, incomodado, aborrecido, ofendido ou humilhado? | [1] Nenhuma vez nos últimos 30 dias: 58.37%<br>[4] 3 ou mais vezes: 19.02%<br>[2] 1 vez: 13.17%<br>[3] 2 vezes: 8.92%<br>[-9] Sem resposta: 0.37%<br>[-2] Abandono: 0.16% |
| NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola se recusou a falar com você, deixou você de lado sem razão ou fez com que outros colegas deixassem de falar com você? | [1] Nenhuma vez nos últimos 30 dias: 72.46%<br>[2] 1 vez: 11.92%<br>[4] 3 ou mais vezes: 9.24%<br>[3] 2 vezes: 5.83%<br>[-9] Sem resposta: 0.39%<br>[-2] Abandono: 0.17% |
| NOS ÚLTIMOS 30 DIAS, você se sentiu ameaçado(a), ofendido(a) ou humilhado(a) nas redes sociais ou aplicativos de celular? | [2] Não: 87.18%<br>[1] Sim: 12.38%<br>[-9] Sem resposta: 0.26%<br>[-2] Abandono: 0.17% |
| NOS ÚLTIMOS 12 MESES, quantas vezes você foi agredido(a) fisicamente por sua mãe, pai ou responsável? | [1] Nenhuma vez nos últimos 12 meses: 77.67%<br>[2] 1 vez: 10.04%<br>[3] 2 a 5 vezes: 7.42%<br>[4] 6 ou mais vezes: 3.75%<br>[-9] Sem resposta: 0.66%<br>[-2] Abandono: 0.46% |
| Alguma vez na vida alguém o(a) tocou, manipulou, beijou ou expôs partes do corpo contra a sua vontade? | [2] Não: 67.17%<br>[-1] Pulo: 15.97%<br>[1] Sim: 15.81%<br>[-9] Sem resposta: 0.58%<br>[-2] Abandono: 0.48% |
| Alguma vez na vida alguém ameaçou, intimidou ou obrigou a ter relações sexuais ou qualquer outro ato sexual contra a sua vontade? | [2] Não: 76.01%<br>[-1] Pulo: 15.97%<br>[1] Sim: 6.93%<br>[-9] Sem resposta: 0.61%<br>[-2] Abandono: 0.48% |
| Qual dessas drogas você usou pela primeira vez? | [-1] Pulo: 93.61%<br>[2] Maconha ou haxixe ou heroína: 5.08%<br>[4] Outra: 0.58%<br>[1] Cocaína ou crack ou merla ou pasta base ou oxi ou metanfetamina ou GHB ou ecstasy: 0.34%<br>[3] LSD ou DMT ou MDMA ou cogumelos alucinógenos: 0.19%<br>[-2] Abandono: 0.12%<br>[-9] Sem resposta: 0.09% |
| NOS ÚLTIMOS 30 DIAS, em quantos dias você fumou cigarros? | [-1]: 85.80%<br>[1] Nenhum dia nos últimos 30 dias: 9.99%<br>[2] 1 ou 2 dias: 1.75%<br>[7] Todos os dias: 0.70%<br>[3] 3 a 5 dias: 0.60%<br>[5] 10 a 19 dias: 0.38%<br>[4] 6 a 9 dias: 0.35%<br>[6] 20 a 29 dias: 0.29%<br>[-2] Abandono: 0.09%<br>[-9] Sem resposta: 0.04% |
| NOS ÚLTIMOS 30 DIAS, em quantos dias você deixou de ir à escola porque não se sentia seguro NO CAMINHO de casa para a escola ou da escola para casa? | [1] Nenhum dia nos últimos 30 dias: 88.63%<br>[2] 1 dia: 3.87%<br>[3] 2 dias: 2.33%<br>[6] 5 dias ou mais: 2.01%<br>[4] 3 dias: 1.41%<br>[5] 4 dias: 0.74%<br>[-9] Sem resposta: 0.61%<br>[-2] Abandono: 0.40% |
| NOS ÚLTIMOS 30 DIAS, em quantos dias você não foi à escola porque não se sentia seguro NA ESCOLA? | [1] Nenhum dia nos últimos 30 dias: 86.86%<br>[2] 1 dia: 4.41%<br>[3] 2 dias: 2.64%<br>[6] 5 dias ou mais: 2.45%<br>[4] 3 dias: 1.69%<br>[5] 4 dias: 0.88%<br>[-9] Sem resposta: 0.66%<br>[-2] Abandono: 0.40% |
| Qual é o seu sexo? | [2] Mulher: 50.05%<br>[1] Homem: 49.95% |
| IDADE DO ESTUDANTE, CATEGORIAS | [2] 13 a 15 anos: 51.61%<br>[3] 16 e 17 anos: 27.76%<br>[1] Menos de 13 anos: 16.03%<br>[4] 18 anos ou mais: 4.52%<br>[-9] Sem resposta: 0.08% |
| Qual é a sua cor ou raça? | [4] Parda: 43.23%<br>[1] Branca: 40.53%<br>[2] Preta: 10.67%<br>[3] Amarela: 2.75%<br>[5] Indígena: 2.25%<br>[-9] Sem resposta: 0.57%<br>[-2] Abandono: 0.01% |
| código de regiões | [2] nordeste: 33.43%<br>[1] norte: 24.48%<br>[3] sudeste: 18.45%<br>[5] centro-oeste: 12.30%<br>[4] sul: 11.33% |
| código UF | [32] Espírito Santo: 4.95%<br>[35] São Paulo: 4.75%<br>[26] Pernambuco: 4.54%<br>[29] Bahia: 4.53%<br>[33] Rio de Janeiro: 4.50%<br>[11] Rondônia: 4.31%<br>[31] Minas Gerais: 4.26%<br>[41] Paraná: 4.15%<br>[17] Tocantins: 4.08%<br>[42] Santa Catarina: 4.03%<br>[50] Mato Grosso do Sul: 3.96%<br>[21] Maranhão: 3.94%<br>[28] Sergipe: 3.85%<br>[27] Alagoas: 3.79%<br>[13] Amazonas: 3.58%<br>[15] Pará: 3.50%<br>[25] Paraíba: 3.49%<br>[22] Piauí: 3.28%<br>[12] Acre: 3.26%<br>[16] Amapá: 3.18%<br>[43] Rio Grande do Sul: 3.14%<br>[23] Ceará: 3.03%<br>[24] Rio Grande do Norte: 2.99%<br>[51] Mato Grosso: 2.95%<br>[52] Goiás: 2.90%<br>[14] Roraima: 2.58%<br>[53] Distrito Federal: 2.49% |
| indica se é município da capital | [1] é capital: 51.21%<br>[0] não é capital: 48.79% |
| Qual nível de ensino (escolaridade) sua MÃE estudou ou estuda? | [7] Terminou o Ensino Superior (faculdade): 37.15%<br>[8] Não sei: 19.68%<br>[5] Terminou o Ensino Médio (2º grau): 16.74%<br>[2] Não terminou o Ensino Fundamental (1º grau): 8.09%<br>[6] Não terminou o Ensino Superior (faculdade): 6.98%<br>[4] Não terminou o Ensino Médio (2º grau): 4.86%<br>[3] Terminou o Ensino Fundamental (1º grau): 3.91%<br>[1] Minha mãe não estudou: 2.37%<br>[-9] Sem resposta: 0.20%<br>[-2] Abandono: 0.03% |
| Na sua casa tem computador ou notebook? | [1] Sim: 63.26%<br>[2] Não: 36.67%<br>[-9] Sem resposta: 0.04%<br>[-2] Abandono: 0.02% |
| Você tem acesso à internet em sua casa? | [1] Sim: 97.26%<br>[2] Não: 2.69%<br>[-9] Sem resposta: 0.03%<br>[-2] Abandono: 0.02% |
| Quantas horas por dia você costuma ficar sentado(a), assistindo televisão, jogando videogame, usando computador, celular, tablet ou fazendo outras atividades sentado(a)? (NÃO contar sábado, domingo, feriados ou o tempo sentado na escola) | [1] Até 1 hora por dia: 14.39%<br>[3] Mais de 2 horas até 3 horas por dia: 13.92%<br>[2] Mais de 1 hora até 2 horas por dia: 13.60%<br>[4] Mais de 3 horas até 4 horas por dia: 12.36%<br>[10] Não sei / não lembro: 10.11%<br>[5] Mais de 4 horas até 5 horas por dia: 9.83%<br>[9] Mais de 8 horas por dia: 8.82%<br>[6] Mais de 5 horas até 6 horas por dia: 7.64%<br>[7] Mais de 6 horas até 7 horas por dia: 5.07%<br>[8] Mais de 7 horas até 8 horas por dia: 3.86%<br>[-9] Sem resposta: 0.31%<br>[-2] Abandono: 0.09% |
| NOS ÚLTIMOS 7 DIAS, sem contar as aulas de educação física da escola, em quantos dias você praticou alguma atividade física? | [1] Nenhum dia nos últimos 7 dias: 28.98%<br>[3] 2 dias: 13.02%<br>[4] 3 dias: 12.34%<br>[6] 5 dias: 12.12%<br>[2] 1 dia: 10.64%<br>[8] 5 dias mais sábado e domingo: 9.36%<br>[5] 4 dias: 8.01%<br>[7] 5 dias mais sábado: 5.22%<br>[-9] Sem resposta: 0.23%<br>[-2] Abandono: 0.08% |

In [11]:
df_limpo.shape

(148707, 30)